In [ ]:
!pip install -U bitsandbytes>=0.46.1 accelerate

# Load BioMed R1-8B Model
Loading the model from huggingface platform followed by initilisationof the
tokenizer used for tokenizing the dataset for the model.

We will be using the Google Colab T4 GPU for inferencing about 50 samples from the PubMedQA Dataset.

This is Zero-Shot Inferencing performed to find out the results of how the model would perform before fine-tuning.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Define the Model ID from Hugging Face
model_id = "zou-lab/BioMed-R1-8B"

# 2. Configure 4-bit Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for stable reasoning
)

# 3. Initialize the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 4. Initialize the Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto", # Automatically assigns to GPU/CPU
    dtype=torch.bfloat16,
    trust_remote_code=True
)

config.json:   0%|          | 0.00/866 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# Load Dataset

In [ ]:
import json

# EXECUTION: Loads the file from your local directory into a Python list of dictionaries.
with open('/content/drive/MyDrive/BioMed AI CW/dev_set.json', 'r') as f:
    test_data = json.load(f)

# LOGIC: 'test_data' now contains objects with 'question', 'context', and 'label'.
print(f"Loaded {len(test_data)} samples for evaluation.")

Loaded 50 samples for evaluation.


# Context Builder
Preprocessing the dataset by transforming unlablled data into a dtructured format which improves the readability of the LLM.



In [ ]:
def build_context(labels: list, contexts: list) -> str:
    """
    BUG 2 FIX: zip section headings (LABELS) with their paragraph text
    (CONTEXTS) so the model sees a structured abstract like:
        BACKGROUND: ...
        METHODS: ...
        RESULTS: ...
    instead of an unlabelled list of strings.
    """
    parts = []
    for label, text in zip(labels, contexts):
        parts.append(f"{label.strip().title()}: {text.strip()}")
    return "\n".join(parts)

# Prompt Engineering


In [ ]:
def get_prompt(question: str, context: str) -> str:
    return (
        f"The following is a structured biomedical abstract:\n\n"
        f"{context}\n\n"
        f"Based ONLY on the evidence in the abstract above, answer this question:\n"
        f"{question}\n\n"
        f"Instructions:\n"
        f"- Think through the evidence step by step.\n"
        f"- Your final answer MUST be exactly one of: yes, no, or maybe.\n"
        f"- You MUST end your response with exactly one of these three lines:\n"
        f"  Final Answer: yes\n"
        f"  Final Answer: no\n"
        f"  Final Answer: maybe\n"
        f"- Do not write anything after the Final Answer line.\n"
    )

In [ ]:
# Phrases the model uses when reasoning towards a conclusion, mapped to labels.
# Drawn from observed model outputs in the unknown cases.
CONCLUSION_SIGNALS = {
    "yes": [
        "therefore yes", "so yes", "answer is yes", "conclude yes",
        "implies yes", "suggests yes", "indicates yes", "supports yes",
        "affirmative", "this is correct", "this is true", "this is supported",
        "effective", "beneficial", "successful", "accurate", "useful",
        "is justified", "is justifiable", "does help", "does work",
        "confirms", "demonstrates", "shows that", "proves",
        "implies affirmative", "the answer should be yes",
    ],
    "no": [
        "therefore no", "so no", "answer is no", "conclude no",
        "implies no", "suggests no", "indicates no", "not supported",
        "not effective", "not beneficial", "not accurate", "not useful",
        "is not justified", "does not help", "does not work",
        "fails to", "no evidence", "no significant", "not significant",
        "the answer should be no",
    ],
    "maybe": [
        "therefore maybe", "so maybe", "answer is maybe", "conclude maybe",
        "unclear", "uncertain", "inconclusive", "mixed evidence",
        "limited evidence", "conflicting", "insufficient evidence",
        "cannot be determined", "hard to say", "it depends",
        "the answer should be maybe",
    ],
}

In [ ]:

def extract_answer(generated_text: str) -> tuple[str, str]:
    """
    Returns (prediction, method) where method is 'formal' or 'fallback'
    so we can track in results how often each path was used.

    Stage 1 — look for explicit 'Final Answer:' marker (first occurrence).
    Stage 2 — scan last 400 chars of reasoning for conclusion signals.
    """
    text = generated_text.lower()

    # ── Stage 1: formal extraction ────────────────────────────────────────────
    if "final answer:" in text:
        answer_part = text.split("final answer:")[1][:30]
        if "yes"   in answer_part: return "yes",   "formal"
        if "no"    in answer_part: return "no",    "formal"
        if "maybe" in answer_part or "unsure" in answer_part:
            return "maybe", "formal"

    # ── Stage 2: fallback — scan tail of reasoning ────────────────────────────
    tail = text[-400:]

    # Score each label by counting how many of its signals appear in the tail
    scores = {label: 0 for label in CONCLUSION_SIGNALS}
    for label, signals in CONCLUSION_SIGNALS.items():
        for signal in signals:
            if signal in tail:
                scores[label] += 1

    best_label = max(scores, key=scores.get)
    if scores[best_label] > 0:
        return best_label, "fallback"

    return "unknown", "unknown"

In [ ]:
from tqdm.auto import tqdm

# ── 6. Inference Loop ─────────────────────────────────────────────────────────
# FIX 1: raised from 512 → 768 to give the model room to conclude
MAX_REASONING_TOKENS = 768

results = []
print(f"\nStarting zero-shot inference on {len(test_data)} samples...")

for entry in tqdm(test_data.values()):
    question     = entry.get('QUESTION',  entry.get('question', ''))
    raw_labels   = entry.get('LABELS',   [])
    raw_contexts = entry.get('CONTEXTS', [])

    if isinstance(raw_contexts, list) and isinstance(raw_labels, list):
        context = build_context(raw_labels, raw_contexts)
    elif isinstance(raw_contexts, list):
        context = " ".join(raw_contexts)
    else:
        context = str(raw_contexts)

    # Correct label field
    gold_label = entry.get('final_decision', '')
    if isinstance(gold_label, list):
        gold_label = gold_label[0]
    gold_label = gold_label.lower().strip()

    prompt = get_prompt(question, context)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_REASONING_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,   # FIX 3: reduced from 1.3 → doesn't block "Final Answer:"
        )

    generated_tokens = outputs[0][input_length:]
    generated_text   = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    prediction, method = extract_answer(generated_text)

    results.append({
        "generated_text": generated_text,
        "gold_label":     gold_label,
        "prediction":     prediction,
        "extract_method": method,   # track formal vs fallback for analysis
    })

with open('zero_shot_results_v3.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to zero_shot_results_v3.json")


Starting zero-shot inference on 50 samples...


  0%|          | 0/50 [00:00<?, ?it/s]

Saved to zero_shot_results_v3.json


In [ ]:

# ── 7. Scoring ────────────────────────────────────────────────────────────────
correct        = 0
pred_counts    = {"yes": 0, "no": 0, "maybe": 0, "unknown": 0}
gold_counts    = {"yes": 0, "no": 0, "maybe": 0}
correct_counts = {"yes": 0, "no": 0, "maybe": 0}
method_counts  = {"formal": 0, "fallback": 0, "unknown": 0}

for r in results:
    pred   = r["prediction"]
    actual = r["gold_label"]

    pred_counts[pred]  = pred_counts.get(pred, 0) + 1
    method_counts[r["extract_method"]] = method_counts.get(r["extract_method"], 0) + 1

    if actual in gold_counts:
        gold_counts[actual] += 1
    if pred == actual:
        correct += 1
        if actual in correct_counts:
            correct_counts[actual] += 1

total    = len(results)
accuracy = (correct / total * 100) if total > 0 else 0.0

print("\n─── Zero-Shot Evaluation Results (v3) ────────────────────────")
print(f"  Model          : {model_id}")
print(f"  Accuracy       : {accuracy:.2f}%  ({correct}/{total} correct)")
print(f"\n  Gold label distribution : {gold_counts}")
print(f"  Prediction breakdown    : {pred_counts}")
print(f"  Per-class correct       : {correct_counts}")
print(f"\n  Extraction method used  : {method_counts}")
print("───────────────────────────────────────────────────────────────\n")


─── Zero-Shot Evaluation Results (v3) ────────────────────────
  Model          : zou-lab/BioMed-R1-8B
  Accuracy       : 78.00%  (39/50 correct)

  Gold label distribution : {'yes': 27, 'no': 17, 'maybe': 6}
  Prediction breakdown    : {'yes': 31, 'no': 16, 'maybe': 0, 'unknown': 3}
  Per-class correct       : {'yes': 25, 'no': 14, 'maybe': 0}

  Extraction method used  : {'formal': 44, 'fallback': 3, 'unknown': 3}
───────────────────────────────────────────────────────────────



In [ ]:
# Per-class Precision / Recall / F1
print("Per-class metrics:")
for cls in ["yes", "no", "maybe"]:
    tp = correct_counts[cls]
    fp = pred_counts[cls] - tp
    fn = gold_counts[cls]  - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    print(f"  {cls:6s} — P: {precision:.2f}  R: {recall:.2f}  F1: {f1:.2f}")

# Fallback accuracy breakdown (good to include in your report)
print("\nAccuracy split by extraction method:")
for method in ["formal", "fallback"]:
    method_results = [r for r in results if r["extract_method"] == method]
    if method_results:
        m_correct = sum(1 for r in method_results if r["prediction"] == r["gold_label"])
        print(f"  {method:8s}: {m_correct}/{len(method_results)} "
              f"= {m_correct/len(method_results)*100:.1f}%")

print("\nSample predictions (first 3):")
for i, r in enumerate(results[:3]):
    print(f"\n[{i+1}] Gold: {r['gold_label']:6s} | Pred: {r['prediction']:7s} "
          f"| Method: {r['extract_method']}")
    print(f"  Last 200 chars: ...{r['generated_text'][-200:]}")

Per-class metrics:
  yes    — P: 0.81  R: 0.93  F1: 0.86
  no     — P: 0.88  R: 0.82  F1: 0.85
  maybe  — P: 0.00  R: 0.00  F1: 0.00

Accuracy split by extraction method:
  formal  : 37/44 = 84.1%
  fallback: 2/3 = 66.7%

Sample predictions (first 3):

[1] Gold: yes    | Pred: yes     | Method: formal
  Last 200 chars: ...

## Final Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer

[2] Gold: yes    | Pred: yes     | Method: formal
  Last 200 chars: ...l Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer: yes

## Final Answer

yes

Final Answer: yes

##

[3] Gold: yes    | Pred: yes     | Method: formal
  Last 200 chars: ...er: yes Final Answer: yes Final Answer: yes Final Answer: yes Final Answer: yes Final Answer: yes Final Answer: yes Final Answer: 

In [ ]:
#Next logits